In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import os.path as osp

DS_PATH = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025"

print(os.listdir(DS_PATH))

TRAIN_PATH = osp.join(DS_PATH, "train/train")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import subprocess
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("git_access_token")

repo_link = "https://" + secret_value_0 + "@github.com/aryansood/CIL.git"

os.chdir("/kaggle/working")

subprocess.run(["rm -rf CIL"], shell=True)
subprocess.run(["git clone -b auto_run " + repo_link], shell=True)

del repo_link

print("repo cloned")
os.chdir("/kaggle/working/CIL")

import wandb
wandb_key = user_secrets.get_secret("wandb_api_key")

wandb.login(
    key=wandb_key
)

['train_list.txt', 'test_list.txt', 'create_prediction_csv.py', 'test', 'train']


Cloning into 'CIL'...


repo cloned


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 10847706 (10847706-ethz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [1]:
import numpy as np 
import pandas as pd
import os
import os.path as osp

TRAIN_PATH = osp.join('/media/luigi/Windows-SSD/Users/luigi/Desktop/CIL/ethz-cil-monocular-depth-estimation-2025', "train/train")

In [ ]:
from training import begin_training_loop
from models.large_unet import UNetMonocularDepthEstimator
from models.unetpp import UNetPlusPlus
import albumentations as A
from pathlib import Path
from datetime import timedelta
import torch

normalize_noise_augmentation = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

augmentations = [
    A.Compose([
        normalize_noise_augmentation,
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=[0.8,1.2], contrast=[0.8,1.2], saturation=[0.8, 1.2], hue=[-0.05, 0.05]),
        A.ToTensorV2()
    ])
]

model = UNetPlusPlus(1e-4, 0.2)


begin_training_loop(
    model = model,
    data_dir = Path(TRAIN_PATH),
    random_split=False,
    train_split=pd.read_csv("train_split.csv")["file_name"].to_list(),
    val_split=pd.read_csv("val_split.csv")["file_name"].to_list(),
    augmentations = augmentations,
    batch_size=2,
    num_worker=4,
    max_training_duration = timedelta(hours=10),
    check_point_every_step = 3000,
    effective_batch_size = 16
)

wandb.finish()

Seed set to 80
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: 10847706 (10847706-ethz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type               | Params | Mode 
----------------------------------------------------
0 | unet | UNetPlusPlusModule | 9.2 M  | train
----------------------------------------------------
9.2 M     Trainable params
0         Non-trainable params
9.2 M     Total params
36.653    Total estimated model params size (MB)
154       Modules in train mode
0         Modules in eval mode


/home/luigi/micromamba/envs/cil/lib/python3.13/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 0:   3%|▎         | 325/9587 [02:05<59:43,  2.58it/s, v_num=vg4a, train_silog_loss=2.540, train_sirme_loss=1.480, grad_2.0_norm/unet.conv0_0.double_conv.0.weight=2.300, grad_2.0_norm/unet.conv0_0.double_conv.0.bias=8.62e-7, grad_2.0_norm/unet.conv0_0.double_conv.1.weight=0.123, grad_2.0_norm/unet.conv0_0.double_conv.1.bias=0.125, grad_2.0_norm/unet.conv0_0.double_conv.4.weight=1.590, grad_2.0_norm/unet.conv0_0.double_conv.4.bias=6.95e-8, grad_2.0_norm/unet.conv0_0.double_conv.5.weight=0.114, grad_2.0_norm/unet.conv0_0.double_conv.5.bias=0.0744, grad_2.0_norm/unet.conv1_0.double_conv.0.weight=1.420, grad_2.0_norm/unet.conv1_0.double_conv.0.bias=3.37e-8, grad_2.0_norm/unet.conv1_0.double_conv.1.weight=0.0801, grad_2.0_norm/unet.conv1_0.double_conv.1.bias=0.0786, grad_2.0_norm/unet.conv1_0.double_conv.4.weight=1.200, grad_2.0_norm/unet.conv1_0.double_conv.4.bias=1.38e-8, grad_2.0_norm/unet.conv1_0.double_conv.5.weight=0.0578, grad_2.0_norm/unet.conv1_0.double_conv.5.bias=0.0612, gra